## 1. Overview

This notebook fine-tunes `meta-llama/Meta-Llama-3.1-8B-Instruct` using the MaxText framework on Kaggle TPU v5e.

- Objective: Run a minimal verification on TPU v5e, then proceed to fine-tuning.
- Evidence of Done: Successful `steps: 1` MaxText run and logs confirming TPU utilization.
- Artifacts: Config file, logs, and checkpoints saved to Kaggle outputs and/or GCS.

Preconditions:
- Kaggle accelerator set to TPU v5e.
- Internet enabled for cloning and dependency installs.
- Access to MaxText-compatible Llama 3.1 checkpoint via Kaggle Datasets.


## 2. Kaggle TPU v5e Environment Setup Plan

Steps in this session:
1. Verify TPU visibility and JAX version
2. Clone MaxText (main branch)
3. Install dependencies from `requirements.txt`
4. Prepare minimal `config.yaml` for verification run
5. Run a 1-step verification to confirm TPU v5e works

Notes:
- No substeps for now; each step maps to a single cell or small group of cells.
- We will capture logs and versions for reproducibility.


In [1]:
# 3. Verify TPU visibility and JAX environment
import os, sys, platform, subprocess, json

print("Python:", sys.version)
print("Platform:", platform.platform())

# Kaggle TPU env vars
for key in ["TPU_NAME", "TPU_WORKER_ID", "TPU_CHIPS_PER_PROCESS", "TPU_MULTISLICE_CTRL_ADDRESS"]:
    if key in os.environ:
        print(f"{key}:", os.environ[key])

try:
    import jax
    import jaxlib
    import jax.numpy as jnp
    print("jax:", jax.__version__)
    print("jaxlib:", jaxlib.__version__)
    devices = jax.devices()
    print("Devices:")
    for d in devices:
        print(" -", d)
    print("Device count:", len(devices))
    x = jnp.ones((8, 8))
    y = jnp.dot(x, x).block_until_ready()
    print("JAX test dot result shape:", y.shape)
except Exception as e:
    print("[ERROR] JAX/TPU verification failed:", e)
    raise


Python: 3.10.18 (main, Jul  1 2025, 05:26:40) [GCC 12.2.0]
Platform: Linux-6.1.42+-x86_64-with-glibc2.36
TPU_WORKER_ID: 0
jax: 0.4.34
jaxlib: 0.4.34


E0000 00:00:1758114821.314512      10 common_lib.cc:612] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:230


Devices:
 - TPU_0(process=0,(0,0,0,0))
 - TPU_1(process=0,(1,0,0,0))
 - TPU_2(process=0,(0,1,0,0))
 - TPU_3(process=0,(1,1,0,0))
 - TPU_4(process=0,(0,2,0,0))
 - TPU_5(process=0,(1,2,0,0))
 - TPU_6(process=0,(0,3,0,0))
 - TPU_7(process=0,(1,3,0,0))
Device count: 8
JAX test dot result shape: (8, 8)


## 4. Clone MaxText (main branch)

We will clone the official `google/maxtext` repository at the default `main` branch for the latest TPU v5e-compatible training scripts. Evidence of done: repository present in the working directory and HEAD commit printed.


In [2]:
%%bash
set -e

echo "Cloning google/maxtext (main)..."
if [ ! -d "maxtext" ]; then
  git clone --depth=1 https://github.com/google/maxtext.git
else
  echo "Repository 'maxtext' already exists; skipping clone."
fi

cd maxtext
echo "Repo HEAD:"
git log -1 --pretty=oneline || true

echo "Top-level files:"
ls -1 | sed -n '1,50p'


Cloning google/maxtext (main)...


Cloning into 'maxtext'...


Repo HEAD:
addd60a6b3a92599a283bfa756b5ffeb1b3a7839 Implement auto-closing of build failure issues after three consecutive successes.
Top-level files:
AUTHORS
CONTRIBUTING.md
LICENSE
PREFLIGHT.md
README.md
RESTRUCTURE.md
benchmarks
clean_py_env.Dockerfile
code_style.sh
docker_build_dependency_image.sh
docker_upload_runner.sh
docs
download_dataset.sh
end_to_end
gpu_multi_process_run.sh
maxtext_custom_wheels.Dockerfile
maxtext_db_dependencies.Dockerfile
maxtext_dependencies.Dockerfile
maxtext_gpu_dependencies.Dockerfile
maxtext_jax_ai_image.Dockerfile
maxtext_libtpu_path.Dockerfile
maxtext_runner.Dockerfile
multihost_job.py
multihost_runner.py
pedagogical_examples
preflight.sh
pylintrc
pyproject.toml
pytest.ini
requirements.txt
requirements_docs.txt
requirements_with_jax_ai_image.txt
requirements_with_jax_stable_stack_0_6_1_pipreqs.txt
rto_setup.sh
setup.sh
setup_gcsfuse.sh
setup_with_retries.sh
src
tests
unit_test_and_lint.sh


## Notes: Handling TensorFlow conflicts on TPU v5e

- Import order: Avoid importing TensorFlow before JAX; it can block TPU init.
- If TF causes conflicts but is not needed for JAX training, consider uninstalling `tensorflow` and using `tensorflow-cpu` instead.
- Keep JAX/jaxlib versions consistent with preinstalled TPU runtime.
- Evidence to capture on failure: exact import stack, package versions, and full error logs.


## 5. Install dependencies from requirements.txt

Install Python dependencies required by MaxText. Kaggle TPU v5e includes a modern JAX stack; if a conflict arises, we will prefer the preinstalled JAX. Evidence of done: successful pip install and import checks.


In [3]:
%%bash
set -e

echo "Updating apt and installing pkg-config..."
apt-get update && apt-get install -y pkg-config

echo "Upgrading pip..."
pip install --upgrade pip

echo "Installing MaxText requirements..."
# Now run the pip install command, which should find the newly installed pkg-config
pip install --no-input --no-cache-dir -r maxtext/requirements.txt

echo "Verifying JAX installation post-install..."
python - <<'PY'
import jax, jaxlib
print("jax:", jax.__version__)
print("jaxlib:", jaxlib.__version__)
print("JAX import successful after requirements install.")
PY

Updating apt and installing pkg-config...
Get:1 http://deb.debian.org/debian bookworm InRelease [151 kB]
Get:2 http://deb.debian.org/debian bookworm-updates InRelease [55.4 kB]
Get:3 http://deb.debian.org/debian-security bookworm-security InRelease [48.0 kB]
Get:4 http://deb.debian.org/debian bookworm/main amd64 Packages [8791 kB]
Get:5 http://deb.debian.org/debian bookworm-updates/main amd64 Packages.diff/Index [21.8 kB]
Ign:5 http://deb.debian.org/debian bookworm-updates/main amd64 Packages.diff/Index
Get:6 http://deb.debian.org/debian-security bookworm-security/main amd64 Packages [278 kB]
Get:7 http://deb.debian.org/debian bookworm-updates/main amd64 Packages [6924 B]
Fetched 9353 kB in 1s (9188 kB/s)
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
pkg-config is already the newest version (1.8.1-1).
pkg-config set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 103 not upgraded.
Upgrading pip...
     ━

Installing MaxText requirements...
     \ 538.6 kB 5.1 MB/s 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
     - 2.8 MB 6.6 MB/s 0:00:000m
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     - 147.1 kB 8.2 MB/s 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
IN

  DEPRECATION: Building 'google-jetstream' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'google-jetstream'. Discussion can be found at https://github.com/pypa/pip/issues/6334


  Created wheel for google-jetstream: filename=google_jetstream-0.3.0-py3-none-any.whl size=166546 sha256=38131196ae2e4596a6f60579be48a6649fabf399c0d73ba77ec1cecb56aa940b
  Stored in directory: /tmp/pip-ephem-wheel-cache-7z3scs7t/wheels/6d/a5/8c/89bbffd4a79016ac6755e4f78e9ad4872de0f001a7fdc17468


  DEPRECATION: Building 'mlperf-logging' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'mlperf-logging'. Discussion can be found at https://github.com/pypa/pip/issues/6334


  Created wheel for mlperf-logging: filename=mlperf_logging-4.1.27-py3-none-any.whl size=321050 sha256=6b7aaea4f497da18c99199635d5862ef788c1074cdca4ade7f213db4d7a8f82a
  Stored in directory: /tmp/pip-ephem-wheel-cache-7z3scs7t/wheels/ba/e6/5e/97c034cf6620d251bb2d641152b8e0e88ba0a43d626124fd48
  Created wheel for qwix: filename=qwix-0.0.0-py3-none-any.whl size=75379 sha256=064e256384259636f036c7e2d7deb4fdeaf89472b150179ffdb55e325b4d68f7
  Stored in directory: /tmp/pip-ephem-wheel-cache-7z3scs7t/wheels/a6/b5/b9/5b17e71886288bfb3a8cef7680371ccbc3198997d8e2f304f3


  DEPRECATION: Building 'antlr4-python3-runtime' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'antlr4-python3-runtime'. Discussion can be found at https://github.com/pypa/pip/issues/6334


  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144591 sha256=2cba6ad688367eeadf5d18110567fcecdca52d7ff27119031de31a6695972c6e
  Stored in directory: /tmp/pip-ephem-wheel-cache-7z3scs7t/wheels/12/93/dd/1f6a127edc45659556564c5730f6d4e300888f4bca2d4c5a88
Successfully built tunix google-jetstream mlperf-logging qwix antlr4-python3-runtime
  Attempting uninstall: keras
    Found existing installation: keras 3.10.0
    Uninstalling keras-3.10.0:
      Successfully uninstalled keras-3.10.0━━━━━   1/119 [keras]━━━━━━━━━━━   1/119 [keras]━━━━━━━━━━━   1/119 [keras]━━━━━━━━━━━   1/119 [keras]━━━━━━━━━━━   1/119 [keras]
  Attempting uninstall: flatbuffers━━━━━━━━━━━   1/119 [keras]━━━━━━━━━━━   1/119 [keras]━━━━━━━━━━━   1/119 [keras]━━━━━━━━━━━   1/119 [keras]━━━━━━━━━━━   1/119 [keras]
    Found existing installation: flatbuffers 25.2.102m  1/119 [keras]
    Uninstalling flatbuffers-25.2.10:━━━━━━━━━   1/119 [keras]
      Successfully uni

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
keras-hub 0.21.1 requires keras>=3.5, but you have keras 2.9.0 which is incompatible.
tensorflow-tpu 2.18.0 requires flatbuffers>=24.3.25, but you have flatbuffers 1.12 which is incompatible.
tensorflow-tpu 2.18.0 requires keras>=3.5.0, but you have keras 2.9.0 which is incompatible.
tensorflow-tpu 2.18.0 requires ml-dtypes<0.5.0,>=0.4.0, but you have ml-dtypes 0.5.3 which is incompatible.
tensorflow-tpu 2.18.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.32.1 which is incompatible.
tensorflow-tpu 2.18.0 requires tensorboard<2.19,>=2.18, but you have tensorboard 2.9.0 which is incompatible.


Verifying JAX installation post-install...
jax: 0.4.34
jaxlib: 0.4.34
JAX import successful after requirements install.


## 6b. Optional: Downgrade NumPy if TensorFlow requires <2

If TensorFlow fails to import with NumPy 2.x, run the next cell to install `numpy<2`. You may need to restart the kernel afterwards for changes to take effect. This is reversible and only applies if needed based on actual errors.


In [4]:
pip install --no-input "numpy<2" && python -c "import numpy as np; print('NumPy version:', np.__version__)"


/usr/local/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 127.0 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
keras-hub 0.21.1 requires keras>=3.5, but you have keras 2.9.0 which is incompatible.
tensorflow-tpu 2.18.0 requires flatbuffers>=24.3.25, but you have flatbuffers 1.12 which is incompatible.
tensorflow-tpu 2.18.0 requires keras>=3.5.0, but you have keras 2.9.0 which is incompatible.
tensorflow-tpu 2.18.0 requires ml-dtypes<0.5.0,>=0.4.0, but you have ml-dtypes 0.5.3 which is incompatible.
tensorflow-tpu 2.18.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.32.1 which is incompatible.
tensorflow-tpu 2.18.0 requires tensorboard

## 6. Configure checkpoint dataset access (Phase 2.3)

This section dynamically detects the Kaggle Dataset path for the pre-converted MaxText checkpoint and exposes it via `MAXTEXT_CHECKPOINT_DIR` for later training steps. No assumptions are made about directory structure.

Evidence of done:
- Dataset directory under `/kaggle/input/...` detected and inspected
- Actual directory contents listed and analyzed
- Checkpoint indicators (`_CHECKPOINT_METADATA`, `items`) verified
- Environment variable `MAXTEXT_CHECKPOINT_DIR` set and printed



In [5]:
# DYNAMIC CHECKPOINT DETECTION - NO ASSUMPTIONS
import os
from pathlib import Path

# Dataset path provided by Kaggle
dataset_path = Path("/kaggle/input/llama-3-1-8b-maxtext-checkpoint")

print(f"🔍 Inspecting dataset directory: {dataset_path}")
print(f"📁 Directory exists: {dataset_path.exists()}")

if not dataset_path.exists():
    raise FileNotFoundError(f"Dataset directory not found: {dataset_path}")

# List all contents to understand actual structure
contents = list(dataset_path.iterdir())
print(f"📋 Directory contents: {[item.name for item in contents]}")

# Look for checkpoint indicators without assumptions
checkpoint_indicators = ["_CHECKPOINT_METADATA", "items", "manifest.ocdbt"]
found_indicators = [indicator for indicator in checkpoint_indicators if (dataset_path / indicator).exists()]

print(f"🔍 Found checkpoint indicators: {found_indicators}")

# Determine checkpoint location based on actual structure
if "_CHECKPOINT_METADATA" in found_indicators and "items" in found_indicators:
    # Checkpoint files are directly in root
    checkpoint_dir = dataset_path
    print(f"✅ Checkpoint found in root directory: {checkpoint_dir}")
elif any((dataset_path / str(i)).exists() for i in range(10)):
    # Look for numbered subdirectories
    numbered_dirs = [d for d in contents if d.is_dir() and d.name.isdigit()]
    if numbered_dirs:
        # Use the first numbered directory found
        checkpoint_dir = numbered_dirs[0]
        print(f"✅ Checkpoint found in numbered subdirectory: {checkpoint_dir}")
    else:
        raise FileNotFoundError("No numbered checkpoint subdirectories found")
else:
    raise FileNotFoundError(f"No recognizable checkpoint structure found. Contents: {[item.name for item in contents]}")

# Verify checkpoint structure
required_files = ["_CHECKPOINT_METADATA", "items"]
missing_files = [f for f in required_files if not (checkpoint_dir / f).exists()]

if missing_files:
    raise FileNotFoundError(f"Checkpoint directory missing required files: {missing_files}")

# Set environment variable (name to be verified with MaxText docs)
os.environ["MAXTEXT_CHECKPOINT_DIR"] = str(checkpoint_dir)

print(f"✅ Verified checkpoint directory: {checkpoint_dir}")
print(f"✅ Environment variable set: MAXTEXT_CHECKPOINT_DIR={os.environ['MAXTEXT_CHECKPOINT_DIR']}")


🔍 Inspecting dataset directory: /kaggle/input/llama-3-1-8b-maxtext-checkpoint
📁 Directory exists: True
📋 Directory contents: ['items', '_CHECKPOINT_METADATA']
🔍 Found checkpoint indicators: ['_CHECKPOINT_METADATA', 'items']
✅ Checkpoint found in root directory: /kaggle/input/llama-3-1-8b-maxtext-checkpoint
✅ Verified checkpoint directory: /kaggle/input/llama-3-1-8b-maxtext-checkpoint
✅ Environment variable set: MAXTEXT_CHECKPOINT_DIR=/kaggle/input/llama-3-1-8b-maxtext-checkpoint


## 7c. Regenerate config with checkpoint loading key

This regenerates the minimal verification YAML and includes the checkpoint load parameter so the model initializes from the MaxText Orbax checkpoint detected above.


In [ ]:
# Generate updated config with checkpoint loading key
import os
from pathlib import Path

checkpoint_path = os.environ.get("MAXTEXT_CHECKPOINT_DIR")
if not checkpoint_path:
    raise ValueError("MAXTEXT_CHECKPOINT_DIR not set. Run checkpoint detection cell first.")

print(f"🔍 Using checkpoint path: {checkpoint_path}")

config_text = f"""# Minimal verification config with checkpoint loading
run_name: "verification_run_1step"
base_output_directory: "/kaggle/working/maxtext_runs"
steps: 1
per_device_batch_size: 1

# Load model parameters from a MaxText Orbax checkpoint directory
load_parameters_path: {checkpoint_path}
""".strip()

out_path = Path("/kaggle/working/verification_minimal.yml")
out_path.write_text(config_text)

print(f"📝 Wrote updated config to: {out_path}")
print("\n--- Generated Config (UPDATED) ---")
print(out_path.read_text())


## 7b. Discover configuration keys for checkpoint/model loading

This cell scans MaxText source and example YAMLs for configuration keys related to checkpoint loading and model selection without assuming parameter names. It prints any candidate keys and example usages for manual verification.


In [6]:
import os, re, glob, yaml
from pathlib import Path

root = Path('maxtext')
candidates = set()

def add_candidates_from_text(text, source):
    # keys that often appear in checkpoint/model config contexts
    patterns = [
        r"load[_-]parameters[_-]path",
        r"load[_-]checkpoint",
        r"checkpoint[_-]path",
        r"restore[_-]path",
        r"init[_-]from",
        r"model",
        r"base[_-]model",
        r"hf[_-]checkpoint",
    ]
    for pat in patterns:
        for m in re.finditer(pat, text, flags=re.IGNORECASE):
            span = text[max(0, m.start()-60): m.end()+60]
            candidates.add((pat, source, span))

# scan yaml files
for yml in root.rglob('*.yml'):
    try:
        txt = yml.read_text(errors='ignore')
        add_candidates_from_text(txt, f"YAML:{yml}")
    except Exception:
        pass

# scan python files
for py in root.rglob('*.py'):
    try:
        txt = py.read_text(errors='ignore')
        add_candidates_from_text(txt, f"PY:{py}")
    except Exception:
        pass

print("Found candidate keys/usages:")
for pat, src, ctx in sorted(candidates):
    print("\n--", pat, "from", src)
    print(ctx)

# Also print top-level YAML keys for exemplar configs if any exist
exemplars = list(root.rglob('*.yml'))[:10]
print("\nSample YAML top-level keys (first 10 files):")
for fp in exemplars:
    try:
        data = yaml.safe_load(fp.read_text())
        if isinstance(data, dict):
            print(fp, list(data.keys())[:20])
    except Exception:
        pass


Found candidate keys/usages:

-- base[_-]model from PY:maxtext/src/MaxText/convert_deepseek_family_ckpt.py

  return mapping


def _convert_huggingface_to_jax_weights(base_model_path, model_params, mem_info, enable_mtp=False) -> dict:
  

-- base[_-]model from PY:maxtext/src/MaxText/convert_deepseek_family_ckpt.py
(1024**3))
  max_logging.log(f"Loading the base model from {base_model_path}")
  return _convert_huggingface_to_jax_weights(base_m

-- base[_-]model from PY:maxtext/src/MaxText/convert_deepseek_family_ckpt.py
, False) and enable_mtp

  ckpt_paths = sorted(pathlib.Path(base_model_path).glob("[!.]*.safetensors"))
  chkpt_vars = {}
  for i,

-- base[_-]model from PY:maxtext/src/MaxText/convert_deepseek_family_ckpt.py
024**3))
  return jax_weights


def _convert_to_jax_weights(base_model_path, model_size, mem_info, enable_mtp=False) -> dict:
  ""

-- base[_-]model from PY:maxtext/src/MaxText/convert_deepseek_family_ckpt.py
_model_path}")
  return _convert_huggingface_to_jax_weigh

## 8b. Invoke training with proper module path

We will ensure `MaxText` is importable by adding `maxtext/src` to `PYTHONPATH`, then invoke `MaxText.train` as a module using the generated YAML.


In [ ]:
# Run training via module with PYTHONPATH set
import os, subprocess, sys

# Mitigate TF protobuf C-extension issues per prior guidance
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

repo_root = "/kaggle/working/maxtext"
package_src = os.path.join(repo_root, "src")
os.environ["PYTHONPATH"] = package_src + (":" + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else "")

train_module = "MaxText.train"
config_path = "/kaggle/working/verification_minimal.yml"

print(f"📚 PYTHONPATH set to include: {package_src}")
print(f"🚀 Running module: {train_module}")
print(f"📄 Using config: {config_path}")

cmd = [sys.executable, "-m", train_module, config_path]
proc = subprocess.run(cmd, capture_output=True, text=True, env=os.environ)

print("=== STDOUT ===")
print(proc.stdout)
if proc.stderr:
    print("=== STDERR ===")
    print(proc.stderr)

if proc.returncode != 0:
    print(f"❌ Verification run failed with return code {proc.returncode}")
else:
    print("✅ Verification run completed successfully.")


## 7. Generate minimal configuration (Phase 2.4)

This section creates a minimal MaxText configuration file using only verified parameters from source code inspection. No assumptions are made about parameter names.

Evidence of done:
- Configuration file generated with verified parameters only
- Unverified parameters clearly marked for documentation review
- Checkpoint path properly integrated


In [7]:
# GENERATE CONFIG WITH ONLY VERIFIED PARAMETERS
import os
from pathlib import Path

# Verify checkpoint path is available
checkpoint_path = os.environ.get("MAXTEXT_CHECKPOINT_DIR")
if not checkpoint_path:
    raise ValueError("MAXTEXT_CHECKPOINT_DIR not set. Run checkpoint detection cell first.")

print(f"🔍 Using checkpoint path: {checkpoint_path}")

# Create minimal config with ONLY verified parameters from source code inspection
# VERIFIED: steps, per_device_batch_size
# UNVERIFIED: All other parameters need to be found in MaxText docs
config_text = f"""# Minimal verification config - ONLY VERIFIED PARAMETERS
# Based on source code inspection of maxtext/src/MaxText/train.py

run_name: "verification_run_1step"
base_output_directory: "/kaggle/working/maxtext_runs"
steps: 1
per_device_batch_size: 1

# TODO: Find correct checkpoint loading parameter name
# Current checkpoint path: {checkpoint_path}
""".strip()

out_path = Path("/kaggle/working/verification_minimal.yml")
out_path.write_text(config_text)

print(f"📝 Wrote config to: {out_path}")
print("\\n--- Generated Config (VERIFIED PARAMETERS ONLY) ---")
print(out_path.read_text())
print("\\n⚠️  CRITICAL: Most parameters need verification - checkpoint loading unknown")


🔍 Using checkpoint path: /kaggle/input/llama-3-1-8b-maxtext-checkpoint
📝 Wrote config to: /kaggle/working/verification_minimal.yml
\n--- Generated Config (VERIFIED PARAMETERS ONLY) ---
# Minimal verification config - ONLY VERIFIED PARAMETERS
# Based on source code inspection of maxtext/src/MaxText/train.py

run_name: "verification_run_1step"
base_output_directory: "/kaggle/working/maxtext_runs"
steps: 1
per_device_batch_size: 1

# TODO: Find correct checkpoint loading parameter name
# Current checkpoint path: /kaggle/input/llama-3-1-8b-maxtext-checkpoint
\n⚠️  CRITICAL: Most parameters need verification - checkpoint loading unknown


## 8. Resolve TensorFlow conflicts and run verification

This section addresses the TensorFlow protobuf conflict using the documented solution and runs the MaxText verification with proper environment setup.

Evidence of done:
- TensorFlow protobuf conflict resolved using documented solution
- All file paths verified before execution
- Process completes with proper error handling and logging

In [8]:
import os, subprocess, sys

# Set environment variable to resolve TensorFlow protobuf conflict
# This is the documented solution from the error message
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

# Verify the train script exists before running
train_script_path = "maxtext/src/MaxText/train.py"
if not os.path.exists(train_script_path):
    raise FileNotFoundError(f"Training script not found: {train_script_path}")

config_path = "/kaggle/working/verification_minimal.yml"
if not os.path.exists(config_path):
    raise FileNotFoundError(f"Config file not found: {config_path}")

print(f"🔧 Set PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python")
print(f"🚀 Running script: {train_script_path}")
print(f"📄 Using config: {config_path}")

# Run with proper environment
cmd = [sys.executable, train_script_path, config_path]
proc = subprocess.run(cmd, capture_output=True, text=True, env=os.environ)

# Print both stdout and stderr for debugging
print("=== STDOUT ===")
print(proc.stdout)
if proc.stderr:
    print("=== STDERR ===")
    print(proc.stderr)

if proc.returncode != 0:
    print(f"❌ Verification run failed with return code {proc.returncode}")
else:
    print("✅ Verification run completed successfully.")

🔧 Set PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python
🚀 Running script: maxtext/src/MaxText/train.py
📄 Using config: /kaggle/working/verification_minimal.yml
=== STDOUT ===

=== STDERR ===
Traceback (most recent call last):
  File "/kaggle/working/maxtext/src/MaxText/train.py", line 45, in <module>
    from MaxText import checkpointing
ModuleNotFoundError: No module named 'MaxText'

❌ Verification run failed with return code 1


## 9. Verify MaxText configuration parameters from source code

This section examines the actual MaxText train.py source code to identify which configuration parameters exist and which were incorrectly assumed.

Evidence of done:
- Source code inspection completed
- Parameter existence verified against actual code
- Assumptions identified and documented

In [9]:
import os
import re
from pathlib import Path

# Find the actual train.py script
train_script_path = "maxtext/src/MaxText/train.py"
if not os.path.exists(train_script_path):
    raise FileNotFoundError(f"MaxText train.py not found: {train_script_path}")

print(f"🔍 Examining MaxText configuration parameters from: {train_script_path}")

# Read the train.py file to find configuration parameters
with open(train_script_path, 'r') as f:
    content = f.read()

# Look for configuration parameter definitions
config_patterns = [
    r'load_parameters_path',
    r'base_model_name', 
    r'model_class',
    r'dataset_type',
    r'steps',
    r'per_device_batch_size'
]

print("\\n📋 Configuration parameters found in MaxText train.py:")
for pattern in config_patterns:
    matches = re.findall(pattern, content)
    if matches:
        print(f"  ✅ {pattern}: {len(matches)} occurrences")
    else:
        print(f"  ❌ {pattern}: NOT FOUND")

# Look for argument parser or config loading
argparse_pattern = r'argparse|ArgumentParser|add_argument'
if re.search(argparse_pattern, content):
    print("\\n🔧 Found argument parsing - checking for specific parameters...")
    
    # Extract argument definitions
    arg_matches = re.findall(r'add_argument\([^)]+\)', content)
    print(f"\\n📝 Found {len(arg_matches)} argument definitions")
    
    for i, arg in enumerate(arg_matches[:10]):  # Show first 10
        print(f"  {i+1}. {arg}")

print("\\n⚠️  Manual verification needed: Check MaxText documentation for correct parameter names")


🔍 Examining MaxText configuration parameters from: maxtext/src/MaxText/train.py
\n📋 Configuration parameters found in MaxText train.py:
  ❌ load_parameters_path: NOT FOUND
  ❌ base_model_name: NOT FOUND
  ❌ model_class: NOT FOUND
  ❌ dataset_type: NOT FOUND
  ✅ steps: 9 occurrences
  ✅ per_device_batch_size: 1 occurrences
\n⚠️  Manual verification needed: Check MaxText documentation for correct parameter names


## 10. Extract actual MaxText configuration parameters

This section extracts the complete list of actual configuration parameters from the MaxText argument parser to replace all assumptions with verified parameter names.

Evidence of done:
- Complete argument parser analysis completed
- All parameter names and help text extracted
- Checkpoint and model-related parameters identified

In [10]:
import os
import re
from pathlib import Path

train_script_path = "maxtext/src/MaxText/train.py"
if not os.path.exists(train_script_path):
    raise FileNotFoundError(f"MaxText train.py not found: {train_script_path}")

print(f"🔍 Extracting actual configuration parameters from: {train_script_path}")

with open(train_script_path, 'r') as f:
    content = f.read()

# Find all argument definitions
arg_pattern = r'add_argument\([^)]+\)'
arg_matches = re.findall(arg_pattern, content)

print(f"\\n📋 Found {len(arg_matches)} argument definitions:")

# Extract parameter names and help text
config_params = []
for i, arg in enumerate(arg_matches):
    # Extract the argument name
    name_match = re.search(r'--([a-zA-Z_][a-zA-Z0-9_-]*)', arg)
    if name_match:
        param_name = name_match.group(1)
        
        # Extract help text if available
        help_match = re.search(r'help=[\'"]([^\'"]*)[\'"]', arg)
        help_text = help_match.group(1) if help_match else "No help text"
        
        config_params.append((param_name, help_text))
        print(f"  {i+1:2d}. --{param_name}")
        print(f"      Help: {help_text}")

# Look for checkpoint-related parameters specifically
checkpoint_params = [p for p in config_params if 'checkpoint' in p[0].lower() or 'load' in p[0].lower() or 'param' in p[0].lower()]
print(f"\\n🔍 Checkpoint-related parameters:")
for name, help_text in checkpoint_params:
    print(f"  --{name}: {help_text}")

# Look for model-related parameters
model_params = [p for p in config_params if 'model' in p[0].lower() or 'base' in p[0].lower()]
print(f"\\n🤖 Model-related parameters:")
for name, help_text in model_params:
    print(f"  --{name}: {help_text}")

print(f"\\n✅ Use these ACTUAL parameter names in configuration")


🔍 Extracting actual configuration parameters from: maxtext/src/MaxText/train.py
\n📋 Found 0 argument definitions:
\n🔍 Checkpoint-related parameters:
\n🤖 Model-related parameters:
\n✅ Use these ACTUAL parameter names in configuration


In [11]:
!find maxtext -name "train.py"

maxtext/src/MaxText/train.py
